<a href="https://colab.research.google.com/github/Zaymerstone/Coupled_Attractor_Neural_Model_Reddish1996/blob/main/Coupled_Attractor_Model_Reddish1996.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install brian2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from brian2 import *

# Use Brian2's runtime mode (easier for development)
prefs.codegen.target = 'numpy'


# MODEL PARAMETERS (from Table 1, Redish et al. 1996)


# --- Simulation ---
dt_sim = 0.1 * ms          # integration time step

# --- Pool sizes ---
N_E = 100                   # number of excitatory units per module
N_I = 100                   # number of inhibitory units per module

# --- Preferred directions (evenly spaced around 360°) ---
phi_E = np.linspace(0, 360, N_E, endpoint=False)  # degrees, for E pools
phi_I = np.linspace(0, 360, N_I, endpoint=False)  # degrees, for I pools

# --- Time constants ---
tau_E = 1.0 * ms            # excitatory unit time constant
tau_I = 0.2 * ms            # inhibitory unit time constant

# --- Tonic inhibition ---
gamma_E = -1.5              # tonic inhibition for E units
gamma_I = -7.5              # tonic inhibition for I units

# --- Gaussian widths for connection profiles ---
sigma_E = 30.0              # degrees, narrow (for E->E and E->I connections)
sigma_I = 360.0             # degrees, broad (for I->I and I->E connections)

# --- Connection weight scaling constants ---
kappa_EE = 5.0              # E -> E (excitatory, narrow)
kappa_IE = 16.0             # E -> I (excitatory, narrow)
kappa_II = -8.0             # I -> I (inhibitory, broad)
kappa_EI = -12.0            # I -> E (inhibitory, broad)

# --- Between-module connection strengths ---
w_PoS_to_ATN_match = 1.0    # PoS:E -> ATN:E matching
w_ATN_to_PoS_match = 0.6    # ATN:E -> PoS:E matching

# --- Offset parameters ---
delta_offset = 10.0         # degrees, offset for left/right connections

# --- Summary printout ---
print("="*50)
print("Redish et al. 1996 - Model Parameters Loaded")
print("="*50)
print(f"Pool size:        {N_E} E units, {N_I} I units per module")
print(f"Preferred dirs:   {phi_E[0]:.1f}° to {phi_E[-1]:.1f}° "
      f"(step = {phi_E[1]-phi_E[0]:.1f}°)")
print(f"Time constants:   tau_E = {tau_E}, tau_I = {tau_I}")
print(f"Tonic inhibition: gamma_E = {gamma_E}, gamma_I = {gamma_I}")
print(f"Gaussian widths:  sigma_E = {sigma_E}°, sigma_I = {sigma_I}°")
print(f"Weight scalars:   EE={kappa_EE}, IE={kappa_IE}, "
      f"II={kappa_II}, EI={kappa_EI}")
print(f"Offset delta:     {delta_offset}°")
print("="*50)

**LOAD THE DATA FROM DRIVE USE BEFORE SESSION**

In [ ]:
#============================================================
# CELL 3 (next session): Load checkpoints + redefine functions
#============================================================
import pickle

checkpoint_dir = '/content/drive/MyDrive/HD_model_extract/checkpoints'

# ---- Load weight matrices ----
with open(f'{checkpoint_dir}/weight_matrices.pkl', 'rb') as f:
    wm = pickle.load(f)
    W_EE = wm['W_EE']
    W_IE = wm['W_IE']
    W_EI = wm['W_EI']
    W_II = wm['W_II']
    phi_E = wm['phi_E']
    phi_I = wm['phi_I']
print("✓ Weight matrices loaded")

# ---- Load model parameters ----
with open(f'{checkpoint_dir}/model_params.pkl', 'rb') as f:
    params = pickle.load(f)
    N_E = params['N_E']
    N_I = params['N_I']
    gamma_E = params['gamma_E']
    gamma_I = params['gamma_I']
    sigma_E = params['sigma_E']
    sigma_I = params['sigma_I']
    kappa_EE = params['kappa_EE']
    kappa_IE = params['kappa_IE']
    kappa_II = params['kappa_II']
    kappa_EI = params['kappa_EI']
    w_PoS_to_ATN_match = params['w_PoS_to_ATN_match']
    w_ATN_to_PoS_match = params['w_ATN_to_PoS_match']
    delta_offset = params['delta_offset']
print("✓ Model parameters loaded")

# ---- Load Step 10 results ----
try:
    with open(f'{checkpoint_dir}/step10_results.pkl', 'rb') as f:
        r10 = pickle.load(f)
        model_time_s = r10['model_time_s']
        bump_pos_model = r10['bump_pos_model']
        true_hd_wrapped = r10['true_hd_wrapped']
        error = r10['error']
        t_start = r10['t_start']
        t_end = r10['t_end']
        vel_interp_deg = r10['vel_interp_deg']
        traj_interp_deg_wrapped = r10['traj_interp_deg_wrapped']
        model_times = r10['model_times']
    print("✓ Step 10 simulation results loaded")
    print(f"  Segment: {t_start:.0f}s to {t_end:.0f}s")
    print(f"  Mean absolute error: {np.mean(np.abs(error)):.1f}°")
except:
    print("✗ Step 10 results not found — need to run simulation")

# ---- Load HD cell analysis ----
try:
    with open(f'{checkpoint_dir}/hd_cell_analysis.pkl', 'rb') as f:
        hd = pickle.load(f)
        hd_candidates = hd['hd_candidates']
    print(f"✓ HD cell analysis loaded ({len(hd_candidates)} candidates)")
except:
    print("✗ HD cell analysis not found")

# ---- Load professor's data ----
data_path = '/content/drive/MyDrive/HD_model_extract/export.pkl'
with open(data_path, 'rb') as fp:
    times, traj, vel, acc = pickle.load(fp)
    units, structs, struct_of_unit, units_by_struct = pickle.load(fp)
    spike_times_of_unit, spike_angles_of_unit = pickle.load(fp)
print("✓ Professor's recording data loaded")

# ---- Redefine all functions ----
def xi(angular_velocity):
    v = np.abs(angular_velocity)
    return v / (v + 200)

def xi_calibrated(angular_velocity):
    v = np.abs(angular_velocity)
    return 0.20 * v / (v + 200)  # calibrated on real data

def population_vector_readout(firing_rates, preferred_directions):
    angles_rad = np.radians(preferred_directions)
    x = np.sum(firing_rates * np.cos(angles_rad))
    y = np.sum(firing_rates * np.sin(angles_rad))
    direction = np.degrees(np.arctan2(y, x)) % 360
    total_firing = np.sum(firing_rates)
    if total_firing > 0:
        strength = np.sqrt(x**2 + y**2) / total_firing
    else:
        strength = 0.0
    return direction, strength

def periodized_gaussian(delta_phi, sigma, N):
    delta_rad = np.radians(delta_phi)
    sigma_rad = np.radians(sigma)
    g = np.exp((np.cos(delta_rad) - 1) / (sigma_rad ** 2))
    g = g / g.sum()
    return g

def rayleigh_vector_length(spike_angles):
    angles = spike_angles % (2 * np.pi)
    if len(angles) == 0:
        return 0, 0
    r = np.abs(np.mean(np.exp(1j * angles)))
    preferred = np.angle(np.mean(np.exp(1j * angles))) % (2 * np.pi)
    return r, preferred

print("✓ All functions redefined")
print("\nReady to continue from where you left off!")

In [ ]:

# for myself -> Gaussian & the Periodization Problem


fig, axes = plt.subplots(2, 2, figsize=(14, 10))


# Plot 1: A simple Gaussian bell curve

x = np.linspace(-180, 180, 1000)
sigma = 30  # degrees
g = np.exp(-x**2 / sigma**2)

axes[0, 0].plot(x, g, 'b-', linewidth=2)
axes[0, 0].set_title('Basic Gaussian (σ = 30°)', fontsize=13)
axes[0, 0].set_xlabel('Difference in preferred direction (°)')
axes[0, 0].set_ylabel('Connection strength')
axes[0, 0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0, 0].annotate('Peak: neurons with\nsame preferred dir',
                     xy=(0, 1), xytext=(60, 0.8),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     fontsize=10, color='red')

# Plot 2: The wrapping problem on a number line

x_full = np.linspace(-400, 400, 2000)
g_full = np.exp(-x_full**2 / sigma**2)

axes[0, 1].plot(x_full, g_full, 'b-', linewidth=2, label='Gaussian centered at 0°')
axes[0, 1].axvline(x=-350, color='red', linewidth=2, linestyle='--', alpha=0.8)
axes[0, 1].axvline(x=10, color='green', linewidth=2, linestyle='--', alpha=0.8)

# Mark the problem
axes[0, 1].annotate('355° vs 5°\nNaive diff = -350°\ng ≈ 0 (WRONG!)',
                     xy=(-350, 0), xytext=(-340, 0.5),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     fontsize=10, color='red')
axes[0, 1].annotate('Correct diff = 10°\ng ≈ 0.89 (RIGHT!)',
                     xy=(10, 0.89), xytext=(80, 0.7),
                     arrowprops=dict(arrowstyle='->', color='green'),
                     fontsize=10, color='green')
axes[0, 1].set_title('The Problem: 355° and 5° look far apart', fontsize=13)
axes[0, 1].set_xlabel('Angular difference (°)')
axes[0, 1].set_ylabel('Connection strength')
axes[0, 1].set_xlim(-400, 400)


# Plot 3: Directions on a circle showing 355° and 5° are close

theta = np.linspace(0, 2*np.pi, 360)
axes[1, 0].plot(np.cos(theta), np.sin(theta), 'k-', linewidth=1)

# Mark 5° and 355°
for angle, label, color in [(5, '5°', 'green'), (355, '355°', 'red')]:
    rad = np.radians(90 - angle)  # convert to math convention
    ax, ay = np.cos(rad), np.sin(rad)
    axes[1, 0].plot(ax, ay, 'o', color=color, markersize=12, zorder=5)
    axes[1, 0].annotate(label, xy=(ax, ay),
                         xytext=(ax*1.3, ay*1.3),
                         fontsize=13, fontweight='bold', color=color,
                         ha='center')

# Draw the short arc between them (10°)
arc_angles = np.linspace(np.radians(90-5), np.radians(90-355+360), 50)
arc_r = 0.85
axes[1, 0].plot(arc_r*np.cos(arc_angles), arc_r*np.sin(arc_angles),
                'purple', linewidth=3)
axes[1, 0].annotate('Only 10° apart!', xy=(0.85, 0.85), xytext=(0.3, 0.5),
                     fontsize=12, color='purple', fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color='purple'))

# Draw the long arc (350°)
arc_long = np.linspace(np.radians(90-5), np.radians(90-355), 200)
axes[1, 0].plot(0.7*np.cos(arc_long), 0.7*np.sin(arc_long),
                'gray', linewidth=2, linestyle='--', alpha=0.5)
axes[1, 0].annotate('350° the long way\n(what naive Gaussian sees)',
                     xy=(-0.5, -0.5), fontsize=9, color='gray',
                     ha='center')

# Mark 0°/360° and other reference directions
for angle, label in [(0, '0°/360°'), (90, '90°'), (180, '180°'), (270, '270°')]:
    rad = np.radians(90 - angle)
    axes[1, 0].annotate(label, xy=(1.1*np.cos(rad), 1.1*np.sin(rad)),
                         fontsize=9, ha='center', va='center', color='gray')

axes[1, 0].set_xlim(-1.5, 1.5)
axes[1, 0].set_ylim(-1.5, 1.5)
axes[1, 0].set_aspect('equal')
axes[1, 0].set_title('On a Circle: 355° and 5° are neighbors!', fontsize=13)
axes[1, 0].axis('off')


# Plot 4: Periodized vs non-periodized Gaussian

phi_diff = np.linspace(0, 360, 1000)

# Non-periodized (broken)
g_naive = np.exp(-phi_diff**2 / sigma**2)

# Periodized with brute force (correct)
g_periodic = np.zeros_like(phi_diff)
for j in range(-5, 6):
    shifted = phi_diff + 360.0 * j
    g_periodic += np.exp(-shifted**2 / sigma**2)

# Periodized with cosine trick (correct, one line)
g_cosine = np.exp((np.cos(np.radians(phi_diff)) - 1) / np.radians(sigma)**2)

axes[1, 1].plot(phi_diff, g_naive, 'r--', linewidth=2,
                label='Plain Gaussian (BROKEN)', alpha=0.7)
axes[1, 1].plot(phi_diff, g_periodic / g_periodic.max(), 'b-', linewidth=2,
                label='Periodized - brute force', alpha=0.7)
axes[1, 1].plot(phi_diff, g_cosine / g_cosine.max(), 'g--', linewidth=2,
                label='Periodized - cosine trick', alpha=0.7)
axes[1, 1].set_title('Periodized vs Non-periodized', fontsize=13)
axes[1, 1].set_xlabel('Preferred direction difference (°)')
axes[1, 1].set_ylabel('Connection strength (normalized)')
axes[1, 1].legend(fontsize=9)
axes[1, 1].annotate('Plain Gaussian dies at edges\n→ units near 0° and 360°\n   are wrongly disconnected',
                     xy=(300, 0.02), fontsize=9, color='red',
                     bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.suptitle('Why We Need Periodized Gaussians for Head Direction',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:

# for myself -> Von Mises Distribution with different κ (concentration) values


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

theta = np.linspace(-180, 180, 1000)
theta_rad = np.radians(theta)

kappas = [0.5, 1, 2, 5, 10, 30]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(kappas)))


# Left: All on same plot for comparison

for k, c in zip(kappas, colors):
    # Von Mises: exp(k * cos(x)) / normalization
    vm = np.exp(k * (np.cos(theta_rad) - 1))  # subtract 1 so peak = 1
    axes[0].plot(theta, vm, linewidth=2, color=c, label=f'κ = {k}')

axes[0].set_title('Von Mises Distribution — Different κ values', fontsize=13)
axes[0].set_xlabel('Angle (°)')
axes[0].set_ylabel('Value (peak-normalized)')
axes[0].legend(fontsize=10)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.3)


# Right: Compare Von Mises to regular Gaussian

sigma = 30  # degrees
sigma_rad = np.radians(sigma)

# Equivalent κ for a Gaussian with this sigma: κ ≈ 1/σ²
k_equiv = 1 / sigma_rad**2

gaussian = np.exp(-theta_rad**2 / sigma_rad**2)
von_mises = np.exp(k_equiv * (np.cos(theta_rad) - 1))

axes[1].plot(theta, gaussian, 'r-', linewidth=2, label=f'Gaussian (σ = {sigma}°)')
axes[1].plot(theta, von_mises, 'b--', linewidth=2,
             label=f'Von Mises (κ = 1/σ² ≈ {k_equiv:.1f})')
axes[1].set_title('Gaussian vs Von Mises (σ = 30°)', fontsize=13)
axes[1].set_xlabel('Angle (°)')
axes[1].set_ylabel('Value')
axes[1].legend(fontsize=10)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.3)
axes[1].annotate('Nearly identical\nnear the peak!',
                 xy=(0, 1), xytext=(80, 0.7),
                 arrowprops=dict(arrowstyle='->', color='green'),
                 fontsize=11, color='green')

plt.suptitle('Von Mises: The "Periodized Gaussian" on a Circle',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:

# STEP 2: Build the Weight Matrices


def periodized_gaussian(delta_phi, sigma, N):
    """
    Compute a periodized normalized Gaussian over angular differences.

    Parameters:
        delta_phi :  the angular differences from equations 7–8
        sigma     : the Gaussian width, sigma E from equation 9
        N         : number of units (for normalization as they use in equation 11)

    Returns:
        Normalized Gaussian values that sum to 1 over N equally spaced points
    """
    # Convert to radians for clean wrapping
    delta_rad = np.radians(delta_phi)
    sigma_rad = np.radians(sigma)
    """

  My first brute force approach, but was bad due to
    for j in range(-5, 6):
        shifted = delta_phi + 360.0 * j
        # Equation 9: Gaussian
        g += np.exp(-shifted**2 / sigma**2)

    """

    # Periodized Gaussian: use von Mises distribuition for wrapping around 360°

    # exp((cos(x) - 1) / sigma^2) approximates a periodized Gaussian
    g = np.exp((np.cos(delta_rad) - 1) / (sigma_rad ** 2))

    # Normalize so that sum over N equally spaced points = 1 eq 12 or 11(dont remember)
    g = g / g.sum()

    return g


def build_weight_matrix(phi_pre, phi_post, sigma, kappa, N_pre):
    """
    Build a weight matrix W where W[i,j] = kappa * g*(phi_post[i] - phi_pre[j])

    Parameters:
        phi_pre  : preferred directions of source (presynaptic) pool
        phi_post : preferred directions of target (postsynaptic) pool
        sigma    : Gaussian width in degrees
        kappa    : scaling constant (sign determines E or I)
        N_pre    : number of presynaptic units (for normalization)

    Returns:
        W : weight matrix, shape (len(phi_post), len(phi_pre))
    """
    W = np.zeros((len(phi_post), len(phi_pre)))
    for i in range(len(phi_post)):
        delta = phi_post[i] - phi_pre  # angular differences
        W[i, :] = kappa * periodized_gaussian(delta, sigma, N_pre)
    return W


# --- Build the four within-module weight matrices ---
# These are the same for both PoS and ATN modules

W_EE = build_weight_matrix(phi_E, phi_E, sigma_E, kappa_EE, N_E)  # E -> E
W_IE = build_weight_matrix(phi_E, phi_I, sigma_E, kappa_IE, N_E)  # E -> I
W_EI = build_weight_matrix(phi_I, phi_E, sigma_I, kappa_EI, N_I)  # I -> E
W_II = build_weight_matrix(phi_I, phi_I, sigma_I, kappa_II, N_I)  # I -> I

print("Weight matrices built:")
print(f"  W_EE (E->E): shape {W_EE.shape}, range [{W_EE.min():.4f}, {W_EE.max():.4f}]")
print(f"  W_IE (E->I): shape {W_IE.shape}, range [{W_IE.min():.4f}, {W_IE.max():.4f}]")
print(f"  W_EI (I->E): shape {W_EI.shape}, range [{W_EI.min():.4f}, {W_EI.max():.4f}]")
print(f"  W_II (I->I): shape {W_II.shape}, range [{W_II.min():.4f}, {W_II.max():.4f}]")

In [ ]:
#============================================================
# Visualize weight matrices to verify correctness
#============================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

matrices = [W_EE, W_IE, W_EI, W_II]
titles = [
    f'E→E (κ={kappa_EE}, σ={sigma_E}°)\nExcitatory, narrow',
    f'E→I (κ={kappa_IE}, σ={sigma_E}°)\nExcitatory, narrow',
    f'I→E (κ={kappa_EI}, σ={sigma_I}°)\nInhibitory, broad',
    f'I→I (κ={kappa_II}, σ={sigma_I}°)\nInhibitory, broad'
]
cmaps = ['Reds', 'Reds', 'Blues_r', 'Blues_r']

for ax, W, title, cmap in zip(axes.flat, matrices, titles, cmaps):
    im = ax.imshow(W, aspect='equal', cmap=cmap,
                   extent=[0, 360, 360, 0])
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Presynaptic preferred dir (°)')
    ax.set_ylabel('Postsynaptic preferred dir (°)')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Within-Module Weight Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#============================================================
# Connection profile from a single unit (unit 0, preferred dir = 0°)
#============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left panel: excitatory connections (from E unit 0)
axes[0].plot(phi_E, W_EE[0, :], 'r-', linewidth=2, label=f'E→E (κ={kappa_EE})')
axes[0].plot(phi_I, W_IE[0, :], 'orange', linewidth=2, label=f'E→I (κ={kappa_IE})')
axes[0].set_title('Connections FROM excitatory unit at 0°')
axes[0].set_xlabel('Target preferred direction (°)')
axes[0].set_ylabel('Connection weight')
axes[0].legend()
axes[0].axhline(y=0, color='k', linewidth=0.5)

# Right panel: inhibitory connections (from I unit 0)
axes[1].plot(phi_E, W_EI[0, :], 'b-', linewidth=2, label=f'I→E (κ={kappa_EI})')
axes[1].plot(phi_I, W_II[0, :], 'cyan', linewidth=2, label=f'I→I (κ={kappa_II})')
axes[1].set_title('Connections FROM inhibitory unit at 0°')
axes[1].set_xlabel('Target preferred direction (°)')
axes[1].set_ylabel('Connection weight')
axes[1].legend()
axes[1].axhline(y=0, color='k', linewidth=0.5)

plt.suptitle('Connection Profiles from a Single Unit', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Problem with Brian2

In [ ]:
#============================================================
# STEP 4: Two Independent Attractor Modules (PoS + ATN)
#============================================================

start_scope()
defaultclock.dt = dt_sim

# ---- Neuron equations (same for all pools) ----
eqs_E = '''
dS/dt = (-S + F) / tau_E_param : 1
F : 1
V : 1
'''

eqs_I = '''
dS/dt = (-S + F) / tau_I_param : 1
F : 1
V : 1
'''

# ---- Create all four pools ----
PoS_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='PoS_E')
PoS_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='PoS_I')
ATN_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='ATN_E')
ATN_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='ATN_I')

# ---- Initialize with random synaptic drives ----
PoS_E.S = np.random.uniform(0, 0.1, N_E)
PoS_I.S = np.random.uniform(0, 0.1, N_I)
ATN_E.S = np.random.uniform(0, 0.1, N_E)
ATN_I.S = np.random.uniform(0, 0.1, N_I)

# ---- Network operation: update V and F for BOTH modules ----
@network_operation(dt=defaultclock.dt)
def update_all():
    # --- PoS module ---
    S_PoS_E = np.array(PoS_E.S[:])
    S_PoS_I = np.array(PoS_I.S[:])

    V_PoS_E = gamma_E + W_EE @ S_PoS_E + W_EI @ S_PoS_I
    V_PoS_I = gamma_I + W_IE @ S_PoS_E + W_II @ S_PoS_I

    F_PoS_E = (1 + np.tanh(V_PoS_E)) / 2
    F_PoS_I = (1 + np.tanh(V_PoS_I)) / 2

    PoS_E.V[:] = V_PoS_E
    PoS_E.F[:] = F_PoS_E
    PoS_I.V[:] = V_PoS_I
    PoS_I.F[:] = F_PoS_I

    # --- ATN module (identical computation, independent for now) ---
    S_ATN_E = np.array(ATN_E.S[:])
    S_ATN_I = np.array(ATN_I.S[:])

    V_ATN_E = gamma_E + W_EE @ S_ATN_E + W_EI @ S_ATN_I
    V_ATN_I = gamma_I + W_IE @ S_ATN_E + W_II @ S_ATN_I

    F_ATN_E = (1 + np.tanh(V_ATN_E)) / 2
    F_ATN_I = (1 + np.tanh(V_ATN_I)) / 2

    ATN_E.V[:] = V_ATN_E
    ATN_E.F[:] = F_ATN_E
    ATN_I.V[:] = V_ATN_I
    ATN_I.F[:] = F_ATN_I

# ---- Monitors for E pools (these are what we care about most) ----
mon_PoS_E = StateMonitor(PoS_E, ['S', 'F'], record=True)
mon_ATN_E = StateMonitor(ATN_E, ['S', 'F'], record=True)

# ---- Build and run ----
net = Network(PoS_E, PoS_I, ATN_E, ATN_I, update_all,
              mon_PoS_E, mon_ATN_E)

print("Running two independent attractor modules for 50 ms...")
net.run(50 * ms, report='text')
print("Done!")

In [ ]:
#============================================================
# Visualize both modules — they should form bumps at DIFFERENT locations
#============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# --- PoS heatmap ---
axes[0, 0].imshow(mon_PoS_E.F[:], aspect='auto', cmap='hot',
                   extent=[0, float(mon_PoS_E.t[-1]/ms), 360, 0],
                   vmin=0, vmax=1)
axes[0, 0].set_title('PoS:E Firing Rates Over Time', fontsize=13)
axes[0, 0].set_xlabel('Time (ms)')
axes[0, 0].set_ylabel('Preferred direction (°)')

# --- ATN heatmap ---
axes[0, 1].imshow(mon_ATN_E.F[:], aspect='auto', cmap='hot',
                   extent=[0, float(mon_ATN_E.t[-1]/ms), 360, 0],
                   vmin=0, vmax=1)
axes[0, 1].set_title('ATN:E Firing Rates Over Time', fontsize=13)
axes[0, 1].set_xlabel('Time (ms)')
axes[0, 1].set_ylabel('Preferred direction (°)')

# --- Final bumps comparison ---
axes[1, 0].plot(phi_E, mon_PoS_E.F[:, -1], 'r-', linewidth=2, label='PoS:E')
axes[1, 0].plot(phi_E, mon_ATN_E.F[:, -1], 'b--', linewidth=2, label='ATN:E')
axes[1, 0].set_title('Final Bumps — Independent (not yet connected)', fontsize=13)
axes[1, 0].set_xlabel('Preferred direction (°)')
axes[1, 0].set_ylabel('Firing rate F')
axes[1, 0].legend(fontsize=12)
axes[1, 0].set_ylim(-0.05, 1.05)

# --- Report ---
pos_peak = phi_E[np.argmax(mon_PoS_E.F[:, -1])]
atn_peak = phi_E[np.argmax(mon_ATN_E.F[:, -1])]

axes[1, 1].text(0.5, 0.6, f'PoS bump at: {pos_peak:.0f}°\nATN bump at: {atn_peak:.0f}°',
                fontsize=18, ha='center', va='center',
                transform=axes[1, 1].transAxes)
axes[1, 1].text(0.5, 0.3, 'Bumps are at DIFFERENT locations\nbecause modules are not connected yet.\nStep 5 will synchronize them.',
                fontsize=12, ha='center', va='center',
                transform=axes[1, 1].transAxes, color='gray')
axes[1, 1].axis('off')
axes[1, 1].set_title('Status', fontsize=13)

plt.suptitle('Step 4: Two Independent Attractor Modules',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"PoS bump peaked at: {pos_peak:.0f}°")
print(f"ATN bump peaked at: {atn_peak:.0f}°")
print(f"Difference: {abs(pos_peak - atn_peak):.0f}° (expected: random, since not connected)")

In [ ]:
#============================================================
# STEP 5: Two Coupled Attractor Modules (Matching Connections)
#============================================================

start_scope()
defaultclock.dt = dt_sim

# ---- Neuron equations ----
eqs_E = '''
dS/dt = (-S + F) / tau_E_param : 1
F : 1
V : 1
'''

eqs_I = '''
dS/dt = (-S + F) / tau_I_param : 1
F : 1
V : 1
'''

# ---- Create all four pools ----
PoS_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='PoS_E')
PoS_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='PoS_I')
ATN_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='ATN_E')
ATN_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='ATN_I')

# ---- Initialize: bumps 30° apart (realistic small offset) ----
for i in range(N_E):
    angle_diff_pos = min(abs(phi_E[i] - 180), 360 - abs(phi_E[i] - 180))
    PoS_E.S[i] = 0.5 * np.exp(-angle_diff_pos**2 / 30**2)

    angle_diff_atn = min(abs(phi_E[i] - 210), 360 - abs(phi_E[i] - 210))
    ATN_E.S[i] = 0.5 * np.exp(-angle_diff_atn**2 / 30**2)

PoS_I.S = np.random.uniform(0, 0.01, N_I)
ATN_I.S = np.random.uniform(0, 0.01, N_I)

# ---- Network operation with matching connections ----
@network_operation(dt=defaultclock.dt)
def update_all():
    S_PoS_E = np.array(PoS_E.S[:])
    S_PoS_I = np.array(PoS_I.S[:])
    S_ATN_E = np.array(ATN_E.S[:])
    S_ATN_I = np.array(ATN_I.S[:])

    # --- PoS voltage (intrinsic + matching from ATN) ---
    V_PoS_E = (gamma_E
               + W_EE @ S_PoS_E
               + W_EI @ S_PoS_I
               + w_ATN_to_PoS_match * S_ATN_E)

    V_PoS_I = (gamma_I
               + W_IE @ S_PoS_E
               + W_II @ S_PoS_I)

    F_PoS_E = (1 + np.tanh(V_PoS_E)) / 2
    F_PoS_I = (1 + np.tanh(V_PoS_I)) / 2

    PoS_E.V[:] = V_PoS_E
    PoS_E.F[:] = F_PoS_E
    PoS_I.V[:] = V_PoS_I
    PoS_I.F[:] = F_PoS_I

    # --- ATN voltage (intrinsic + matching from PoS) ---
    V_ATN_E = (gamma_E
               + W_EE @ S_ATN_E
               + W_EI @ S_ATN_I
               + w_PoS_to_ATN_match * S_PoS_E)

    V_ATN_I = (gamma_I
               + W_IE @ S_ATN_E
               + W_II @ S_ATN_I)

    F_ATN_E = (1 + np.tanh(V_ATN_E)) / 2
    F_ATN_I = (1 + np.tanh(V_ATN_I)) / 2

    ATN_E.V[:] = V_ATN_E
    ATN_E.F[:] = F_ATN_E
    ATN_I.V[:] = V_ATN_I
    ATN_I.F[:] = F_ATN_I

# ---- Monitors ----
mon_PoS_E = StateMonitor(PoS_E, ['S', 'F'], record=True)
mon_ATN_E = StateMonitor(ATN_E, ['S', 'F'], record=True)

# ---- Build and run ----
net = Network(PoS_E, PoS_I, ATN_E, ATN_I, update_all,
              mon_PoS_E, mon_ATN_E)

print("Running coupled modules for 50 ms...")
print("  - PoS initialized at 180°, ATN initialized at 210°")
print("  - 30° apart — matching connections should synchronize them")
net.run(50 * ms, report='text')
print("Done!")

In [ ]:
#============================================================
# Visualize: do matching connections synchronize the two bumps?
#============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- PoS heatmap ---
axes[0, 0].imshow(mon_PoS_E.F[:], aspect='auto', cmap='hot',
                   extent=[0, float(mon_PoS_E.t[-1]/ms), 360, 0],
                   vmin=0, vmax=1)
axes[0, 0].set_title('PoS:E — started at 90°', fontsize=13)
axes[0, 0].set_xlabel('Time (ms)')
axes[0, 0].set_ylabel('Preferred direction (°)')

# --- ATN heatmap ---
axes[0, 1].imshow(mon_ATN_E.F[:], aspect='auto', cmap='hot',
                   extent=[0, float(mon_ATN_E.t[-1]/ms), 360, 0],
                   vmin=0, vmax=1)
axes[0, 1].set_title('ATN:E — started at 270°', fontsize=13)
axes[0, 1].set_xlabel('Time (ms)')
axes[0, 1].set_ylabel('Preferred direction (°)')

# --- Snapshots at key moments ---
times_to_plot = [0, 2, 5, 10, 20, 49]
colors_t = plt.cm.viridis(np.linspace(0, 1, len(times_to_plot)))

for t_ms, c in zip(times_to_plot, colors_t):
    t_idx = int(t_ms * ms / defaultclock.dt)
    if t_idx < len(mon_PoS_E.t):
        axes[1, 0].plot(phi_E, mon_PoS_E.F[:, t_idx], '-', color=c,
                        linewidth=2, label=f't={t_ms}ms')
axes[1, 0].set_title('PoS:E Bump Over Time', fontsize=13)
axes[1, 0].set_xlabel('Preferred direction (°)')
axes[1, 0].set_ylabel('Firing rate F')
axes[1, 0].legend(fontsize=9)
axes[1, 0].set_ylim(-0.05, 1.05)

# --- Final state comparison ---
axes[1, 1].plot(phi_E, mon_PoS_E.F[:, -1], 'r-', linewidth=2, label='PoS:E (final)')
axes[1, 1].plot(phi_E, mon_ATN_E.F[:, -1], 'b--', linewidth=2, label='ATN:E (final)')
axes[1, 1].set_title('Final State — Are they synchronized?', fontsize=13)
axes[1, 1].set_xlabel('Preferred direction (°)')
axes[1, 1].set_ylabel('Firing rate F')
axes[1, 1].legend(fontsize=12)
axes[1, 1].set_ylim(-0.05, 1.05)

pos_peak = phi_E[np.argmax(mon_PoS_E.F[:, -1])]
atn_peak = phi_E[np.argmax(mon_ATN_E.F[:, -1])]
axes[1, 1].axvline(x=pos_peak, color='r', linestyle=':', alpha=0.5)
axes[1, 1].axvline(x=atn_peak, color='b', linestyle=':', alpha=0.5)

plt.suptitle('Step 5: Matching Connections — Bump Synchronization',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"PoS final bump at: {pos_peak:.0f}°")
print(f"ATN final bump at: {atn_peak:.0f}°")
print(f"Difference: {abs(pos_peak - atn_peak):.0f}°")
print(f"\nStarted 180° apart → ended {abs(pos_peak - atn_peak):.0f}° apart")

In [ ]:
#============================================================
# STEP 6: The ξ (xi) function — angular velocity to connection strength
#============================================================

def xi(angular_velocity):
    """
    Maps angular head velocity (degrees/second) to offset connection strength.

    Based on Figure 4 of Redish et al. 1996.
    Uses a saturating function that approximates the curve in the paper:
    - 0 velocity → 0 strength
    - Rises steeply at first
    - Saturates near ~0.9 for high velocities

    Parameters:
        angular_velocity : angular speed in degrees/second (always positive)

    Returns:
        Connection strength between 0 and ~0.9
    """
    v = np.abs(angular_velocity)
    return v / (v + 200)


# ---- Visualize and compare to Figure 4 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

velocities = np.linspace(0, 600, 1000)

# Left panel: our ξ function
axes[0].plot(velocities, xi(velocities), 'b-', linewidth=2)
axes[0].set_title('Our ξ(φ̇) Function', fontsize=13)
axes[0].set_xlabel('Angular velocity (degrees/second)')
axes[0].set_ylabel('ξ — offset connection strength')
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.3)

# Mark some key points
for v in [50, 100, 200, 400, 600]:
    axes[0].plot(v, xi(v), 'ro', markersize=6)
    axes[0].annotate(f'{v}°/s → {xi(v):.2f}',
                     xy=(v, xi(v)), xytext=(v+20, xi(v)-0.08),
                     fontsize=9)

# Right panel: show what different velocities mean physically
axes[1].barh(['Stationary', 'Slow turn\n(50°/s)', 'Moderate turn\n(200°/s)',
              'Fast turn\n(400°/s)', 'Very fast turn\n(600°/s)'],
             [xi(0), xi(50), xi(200), xi(400), xi(600)],
             color=['gray', 'lightblue', 'steelblue', 'royalblue', 'darkblue'])
axes[1].set_xlabel('ξ — offset connection strength')
axes[1].set_title('Physical Meaning of ξ Values', fontsize=13)
axes[1].set_xlim(0, 1)

for i, v in enumerate([0, 50, 200, 400, 600]):
    axes[1].text(xi(v) + 0.02, i, f'{xi(v):.2f}', va='center', fontsize=11)

plt.suptitle('Step 6: The ξ Function — Velocity to Connection Strength',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Print summary
print("ξ function behavior:")
print(f"  Stationary (0°/s):     ξ = {xi(0):.3f} (no offset input)")
print(f"  Slow turn (50°/s):     ξ = {xi(50):.3f}")
print(f"  Moderate turn (200°/s): ξ = {xi(200):.3f}")
print(f"  Fast turn (400°/s):    ξ = {xi(400):.3f}")
print(f"  Very fast (600°/s):    ξ = {xi(600):.3f} (nearly saturated)")

In [ ]:
#============================================================
# STEP 7: Full Coupled Model — Offset Connections + Gain Control
#============================================================

start_scope()
defaultclock.dt = dt_sim

# ---- Neuron equations ----
eqs_E = '''
dS/dt = (-S + F) / tau_E_param : 1
F : 1
V : 1
'''

eqs_I = '''
dS/dt = (-S + F) / tau_I_param : 1
F : 1
V : 1
'''

# ---- Create all four pools ----
PoS_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='PoS_E')
PoS_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='PoS_I')
ATN_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='ATN_E')
ATN_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='ATN_I')

# ---- Precompute offset index mappings ----
# For right offset: PoS unit i connects to ATN unit at (i + offset_steps) % N_E
# For left offset:  PoS unit i connects to ATN unit at (i - offset_steps) % N_E
offset_steps = int(round(delta_offset / (360.0 / N_E)))  # 10° / 3.6° ≈ 3 units
print(f"Offset = {delta_offset}° = {offset_steps} units")

right_offset_indices = np.array([(i + offset_steps) % N_E for i in range(N_E)])
left_offset_indices = np.array([(i - offset_steps) % N_E for i in range(N_E)])

# ---- Angular velocity input (will be changed in Step 9) ----
# For now: constant rightward rotation at 200°/s
angular_velocity = 200.0  # degrees/second, positive = rightward

# ---- Initialize bumps at same location (180°) ----
for i in range(N_E):
    angle_diff = min(abs(phi_E[i] - 180), 360 - abs(phi_E[i] - 180))
    bump_val = 0.5 * np.exp(-angle_diff**2 / 30**2)
    PoS_E.S[i] = bump_val
    ATN_E.S[i] = bump_val

PoS_I.S = np.random.uniform(0, 0.01, N_I)
ATN_I.S = np.random.uniform(0, 0.01, N_I)

# ---- Network operation: FULL MODEL ----
@network_operation(dt=defaultclock.dt)
def update_all():
    S_PoS_E = np.array(PoS_E.S[:])
    S_PoS_I = np.array(PoS_I.S[:])
    S_ATN_E = np.array(ATN_E.S[:])
    S_ATN_I = np.array(ATN_I.S[:])

    # --- Compute offset and gain control signals ---
    xi_val = xi(angular_velocity)
    gain_control = -0.5 * xi_val  # mammillary body inhibition

    # Determine which offset connections are active
    if angular_velocity > 0:  # rightward turn
        offset_input = xi_val * S_PoS_E[left_offset_indices]
    elif angular_velocity < 0:  # leftward turn
        offset_input = xi_val * S_PoS_E[right_offset_indices]
    else:
        offset_input = np.zeros(N_E)

    # --- PoS voltage ---
    V_PoS_E = (gamma_E
               + W_EE @ S_PoS_E            # intrinsic E->E
               + W_EI @ S_PoS_I            # intrinsic I->E
               + w_ATN_to_PoS_match * S_ATN_E)  # matching from ATN

    V_PoS_I = (gamma_I
               + W_IE @ S_PoS_E            # intrinsic E->I
               + W_II @ S_PoS_I)           # intrinsic I->I

    F_PoS_E = (1 + np.tanh(V_PoS_E)) / 2
    F_PoS_I = (1 + np.tanh(V_PoS_I)) / 2

    PoS_E.V[:] = V_PoS_E
    PoS_E.F[:] = F_PoS_E
    PoS_I.V[:] = V_PoS_I
    PoS_I.F[:] = F_PoS_I

    # --- ATN voltage ---
    V_ATN_E = (gamma_E + gain_control       # tonic inhibition + MB gain control
               + W_EE @ S_ATN_E             # intrinsic E->E
               + W_EI @ S_ATN_I             # intrinsic I->E
               + w_PoS_to_ATN_match * S_PoS_E   # matching from PoS
               + offset_input)              # offset connections from PoS

    V_ATN_I = (gamma_I
               + W_IE @ S_ATN_E             # intrinsic E->I
               + W_II @ S_ATN_I)            # intrinsic I->I

    F_ATN_E = (1 + np.tanh(V_ATN_E)) / 2
    F_ATN_I = (1 + np.tanh(V_ATN_I)) / 2

    ATN_E.V[:] = V_ATN_E
    ATN_E.F[:] = F_ATN_E
    ATN_I.V[:] = V_ATN_I
    ATN_I.F[:] = F_ATN_I

# ---- Monitors ----
mon_PoS_E = StateMonitor(PoS_E, ['S', 'F'], record=True)
mon_ATN_E = StateMonitor(ATN_E, ['S', 'F'], record=True)

# ---- Build and run ----
net = Network(PoS_E, PoS_I, ATN_E, ATN_I, update_all,
              mon_PoS_E, mon_ATN_E)

print(f"Running full model with constant rotation = {angular_velocity}°/s")
print(f"  ξ({angular_velocity}) = {xi(angular_velocity):.3f}")
print(f"  Gain control = {-0.5 * xi(angular_velocity):.3f}")
print(f"  Bump starts at 180°, should move rightward")
net.run(200 * ms, report='text')
print("Done!")

In [ ]:
#============================================================
# Visualize the bump MOVING under constant angular velocity
#============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- PoS heatmap ---
im1 = axes[0, 0].imshow(mon_PoS_E.F[:], aspect='auto', cmap='hot',
                   extent=[0, float(mon_PoS_E.t[-1]/ms), 360, 0],
                   vmin=0, vmax=1)
axes[0, 0].set_title('PoS:E — should see bump drifting rightward', fontsize=12)
axes[0, 0].set_xlabel('Time (ms)')
axes[0, 0].set_ylabel('Preferred direction (°)')
plt.colorbar(im1, ax=axes[0, 0])

# --- ATN heatmap ---
im2 = axes[0, 1].imshow(mon_ATN_E.F[:], aspect='auto', cmap='hot',
                   extent=[0, float(mon_ATN_E.t[-1]/ms), 360, 0],
                   vmin=0, vmax=1)
axes[0, 1].set_title('ATN:E — should lead PoS slightly', fontsize=12)
axes[0, 1].set_xlabel('Time (ms)')
axes[0, 1].set_ylabel('Preferred direction (°)')
plt.colorbar(im2, ax=axes[0, 1])

# --- Snapshots of PoS bump at different times ---
total_time_ms = float(mon_PoS_E.t[-1] / ms)
times_to_plot = [0, 20, 50, 100, 150, int(total_time_ms)-1]
colors_t = plt.cm.viridis(np.linspace(0, 1, len(times_to_plot)))

for t_ms, c in zip(times_to_plot, colors_t):
    t_idx = int(t_ms * ms / defaultclock.dt)
    if t_idx < len(mon_PoS_E.t):
        axes[1, 0].plot(phi_E, mon_PoS_E.F[:, t_idx], '-', color=c,
                        linewidth=2, label=f't={t_ms}ms')
axes[1, 0].set_title('PoS:E Bump Snapshots — Moving!', fontsize=13)
axes[1, 0].set_xlabel('Preferred direction (°)')
axes[1, 0].set_ylabel('Firing rate F')
axes[1, 0].legend(fontsize=9)
axes[1, 0].set_ylim(-0.05, 1.05)

# --- Track bump position over time ---
# Compute population vector at each timestep
bump_positions_PoS = []
bump_positions_ATN = []
for t_idx in range(len(mon_PoS_E.t)):
    F_pos = mon_PoS_E.F[:, t_idx]
    F_atn = mon_ATN_E.F[:, t_idx]

    # Population vector (weighted circular mean)
    angle_rad = np.radians(phi_E)

    pos_x = np.sum(F_pos * np.cos(angle_rad))
    pos_y = np.sum(F_pos * np.sin(angle_rad))
    bump_positions_PoS.append(np.degrees(np.arctan2(pos_y, pos_x)) % 360)

    atn_x = np.sum(F_atn * np.cos(angle_rad))
    atn_y = np.sum(F_atn * np.sin(angle_rad))
    bump_positions_ATN.append(np.degrees(np.arctan2(atn_y, atn_x)) % 360)

time_ms = mon_PoS_E.t / ms

axes[1, 1].plot(time_ms, bump_positions_PoS, 'r-', linewidth=2, label='PoS:E bump')
axes[1, 1].plot(time_ms, bump_positions_ATN, 'b--', linewidth=1.5, label='ATN:E bump')
axes[1, 1].set_title('Bump Position Over Time', fontsize=13)
axes[1, 1].set_xlabel('Time (ms)')
axes[1, 1].set_ylabel('Represented direction (°)')
axes[1, 1].legend(fontsize=11)

plt.suptitle(f'Step 7: Bump Moving Under Constant Rotation ({angular_velocity}°/s rightward)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nPoS bump: started at ~180°, ended at ~{bump_positions_PoS[-1]:.0f}°")
print(f"ATN bump: started at ~180°, ended at ~{bump_positions_ATN[-1]:.0f}°")

VIZUALIZATION

In [ ]:
#============================================================
# ANIMATED visualization — watch the bump move in real time
#============================================================
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- We already ran the simulation in Cell 13 ---
# --- Now we animate the recorded data ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# How many frames to show (subsample for smooth animation)
# The simulation has many timesteps, we take every Nth one
total_steps = len(mon_PoS_E.t)
frame_skip = max(1, total_steps // 200)  # ~200 frames total
frame_indices = range(0, total_steps, frame_skip)
n_frames = len(list(frame_indices))

# Left panel: bump profile
line_pos, = axes[0].plot(phi_E, mon_PoS_E.F[:, 0], 'r-', linewidth=2, label='PoS:E')
line_atn, = axes[0].plot(phi_E, mon_ATN_E.F[:, 0], 'b--', linewidth=2, label='ATN:E')
axes[0].set_xlim(0, 360)
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_xlabel('Preferred direction (°)', fontsize=12)
axes[0].set_ylabel('Firing rate F', fontsize=12)
axes[0].legend(fontsize=11, loc='upper right')
axes[0].set_title('Bump Profile', fontsize=13)
time_text = axes[0].text(0.02, 0.95, '', transform=axes[0].transAxes,
                          fontsize=13, fontweight='bold', va='top')

# Right panel: bump position trail over time
axes[1].set_xlim(0, float(mon_PoS_E.t[-1]/ms))
axes[1].set_ylim(0, 360)
axes[1].set_xlabel('Time (ms)', fontsize=12)
axes[1].set_ylabel('Represented direction (°)', fontsize=12)
axes[1].set_title('Bump Position Over Time', fontsize=13)
trail_pos, = axes[1].plot([], [], 'r-', linewidth=2, label='PoS:E')
trail_atn, = axes[1].plot([], [], 'b--', linewidth=1.5, label='ATN:E')
axes[1].legend(fontsize=11)

# Precompute all bump positions
all_pos_positions = []
all_atn_positions = []
angle_rad = np.radians(phi_E)

for t_idx in range(total_steps):
    F_pos = mon_PoS_E.F[:, t_idx]
    F_atn = mon_ATN_E.F[:, t_idx]

    px = np.sum(F_pos * np.cos(angle_rad))
    py = np.sum(F_pos * np.sin(angle_rad))
    all_pos_positions.append(np.degrees(np.arctan2(py, px)) % 360)

    ax = np.sum(F_atn * np.cos(angle_rad))
    ay = np.sum(F_atn * np.sin(angle_rad))
    all_atn_positions.append(np.degrees(np.arctan2(ay, ax)) % 360)

time_array = np.array(mon_PoS_E.t / ms)

plt.suptitle(f'Bump Moving in Real Time — {angular_velocity}°/s Rightward',
             fontsize=15, fontweight='bold')
plt.tight_layout()

def animate(frame):
    t_idx = frame * frame_skip
    if t_idx >= total_steps:
        t_idx = total_steps - 1

    # Update bump profiles
    line_pos.set_ydata(mon_PoS_E.F[:, t_idx])
    line_atn.set_ydata(mon_ATN_E.F[:, t_idx])

    # Update time label
    current_time = time_array[t_idx]
    time_text.set_text(f't = {current_time:.1f} ms')

    # Update trails (show history up to current time)
    trail_pos.set_data(time_array[:t_idx], all_pos_positions[:t_idx])
    trail_atn.set_data(time_array[:t_idx], all_atn_positions[:t_idx])

    return line_pos, line_atn, time_text, trail_pos, trail_atn

anim = FuncAnimation(fig, animate, frames=n_frames, interval=50, blit=True)
plt.close()  # prevent static display
HTML(anim.to_jshtml())

In [ ]:
#============================================================
# STEP 8: Population Vector Readout Function
#============================================================

def population_vector_readout(firing_rates, preferred_directions):
    """
    Compute the represented head direction from population activity
    using the weighted circular mean (population vector).

    This is the decoding method described in the paper's introduction:
    the direction of the weighted vector sum of unit vectors
    pointing in each neuron's preferred direction.

    Parameters:
        firing_rates         : array of firing rates (N neurons)
        preferred_directions : array of preferred directions in degrees (N neurons)

    Returns:
        direction : represented direction in degrees (0-360)
        strength  : length of the population vector (0-1ish,
                    indicates how concentrated the bump is)
    """
    angles_rad = np.radians(preferred_directions)

    # Weighted sum of unit vectors
    x = np.sum(firing_rates * np.cos(angles_rad))
    y = np.sum(firing_rates * np.sin(angles_rad))

    # Direction: angle of the resultant vector
    direction = np.degrees(np.arctan2(y, x)) % 360

    # Strength: length of resultant, normalized by total firing
    total_firing = np.sum(firing_rates)
    if total_firing > 0:
        strength = np.sqrt(x**2 + y**2) / total_firing
    else:
        strength = 0.0

    return direction, strength


# ---- Test it on the data from our simulation ----
# Check a few timepoints
print("Testing population vector readout on Step 7 simulation:\n")
print(f"{'Time (ms)':>12}  {'PoS dir':>10}  {'ATN dir':>10}  {'ATN lead':>10}  {'Strength':>10}")
print("-" * 60)

test_times = [0, 25, 50, 100, 150, 199]
for t_ms in test_times:
    t_idx = int(t_ms * ms / defaultclock.dt)
    if t_idx >= len(mon_PoS_E.t):
        t_idx = len(mon_PoS_E.t) - 1

    pos_dir, pos_str = population_vector_readout(mon_PoS_E.F[:, t_idx], phi_E)
    atn_dir, atn_str = population_vector_readout(mon_ATN_E.F[:, t_idx], phi_E)

    # ATN lead (how far ahead ATN is)
    lead = atn_dir - pos_dir
    if lead > 180: lead -= 360
    if lead < -180: lead += 360

    print(f"{t_ms:>12}  {pos_dir:>9.1f}°  {atn_dir:>9.1f}°  {lead:>+9.1f}°  {pos_str:>9.3f}")

print(f"\nATN consistently leads PoS — this matches the paper's prediction")
print(f"that ATN represents future head direction while PoS represents current.")

In [ ]:
#============================================================
# STEP 9: Drive with varying angular velocity inputs
#============================================================

start_scope()
defaultclock.dt = dt_sim

# ---- Define a time-varying angular velocity signal ----
# Total duration: 500 ms
# Scenario: rightward → stop → leftward → fast rightward → stop
total_duration = 500  # ms
time_points = np.arange(0, total_duration, float(dt_sim/ms))  # in ms

velocity_signal = np.zeros_like(time_points)

# Phase 1 (0-100 ms):    Rightward rotation at 150°/s
velocity_signal[(time_points >= 0) & (time_points < 100)] = 150.0

# Phase 2 (100-150 ms):  Stop — no rotation
velocity_signal[(time_points >= 100) & (time_points < 150)] = 0.0

# Phase 3 (150-300 ms):  Leftward rotation at -200°/s
velocity_signal[(time_points >= 150) & (time_points < 300)] = -200.0

# Phase 4 (300-400 ms):  Fast rightward rotation at 400°/s
velocity_signal[(time_points >= 300) & (time_points < 400)] = 400.0

# Phase 5 (400-500 ms):  Stop
velocity_signal[(time_points >= 400) & (time_points < 500)] = 0.0

# Store as a global array that update_all can index into
current_step = [0]  # mutable container so network_operation can modify it

# ---- Visualize the velocity input signal ----
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(time_points, velocity_signal, 'k-', linewidth=2)
ax.fill_between(time_points, velocity_signal, 0,
                where=velocity_signal > 0, alpha=0.3, color='blue', label='Rightward')
ax.fill_between(time_points, velocity_signal, 0,
                where=velocity_signal < 0, alpha=0.3, color='red', label='Leftward')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Angular velocity (°/s)', fontsize=12)
ax.set_title('Input Signal — Angular Velocity Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Velocity scenario:")
print("  0-100 ms:   Rightward at 150°/s")
print("  100-150 ms: Stop")
print("  150-300 ms: Leftward at -200°/s")
print("  300-400 ms: Fast rightward at 400°/s")
print("  400-500 ms: Stop")

In [ ]:
#============================================================
# Run the full model with time-varying velocity
#============================================================

# ---- Neuron equations ----
eqs_E = '''
dS/dt = (-S + F) / tau_E_param : 1
F : 1
V : 1
'''

eqs_I = '''
dS/dt = (-S + F) / tau_I_param : 1
F : 1
V : 1
'''

# ---- Create all four pools ----
PoS_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='PoS_E')
PoS_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='PoS_I')
ATN_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='ATN_E')
ATN_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='ATN_I')

# ---- Offset index mappings ----
offset_steps = int(round(delta_offset / (360.0 / N_E)))
right_offset_indices = np.array([(i + offset_steps) % N_E for i in range(N_E)])
left_offset_indices = np.array([(i - offset_steps) % N_E for i in range(N_E)])

# ---- Initialize bumps at 180° ----
for i in range(N_E):
    angle_diff = min(abs(phi_E[i] - 180), 360 - abs(phi_E[i] - 180))
    bump_val = 0.5 * np.exp(-angle_diff**2 / 30**2)
    PoS_E.S[i] = bump_val
    ATN_E.S[i] = bump_val

PoS_I.S = np.random.uniform(0, 0.01, N_I)
ATN_I.S = np.random.uniform(0, 0.01, N_I)

# ---- Network operation with time-varying velocity ----
@network_operation(dt=defaultclock.dt)
def update_all():
    S_PoS_E = np.array(PoS_E.S[:])
    S_PoS_I = np.array(PoS_I.S[:])
    S_ATN_E = np.array(ATN_E.S[:])
    S_ATN_I = np.array(ATN_I.S[:])

    # Get current angular velocity from the signal
    step = current_step[0]
    if step < len(velocity_signal):
        ang_vel = velocity_signal[step]
    else:
        ang_vel = 0.0
    current_step[0] += 1

    # Compute offset and gain control
    xi_val = xi(ang_vel)
    gain_control = -0.5 * xi_val

    # Offset connections
    if ang_vel > 0:  # rightward
        offset_input = xi_val * S_PoS_E[left_offset_indices]
    elif ang_vel < 0:  # leftward
        offset_input = xi_val * S_PoS_E[right_offset_indices]
    else:
        offset_input = np.zeros(N_E)

    # --- PoS voltage ---
    V_PoS_E = (gamma_E
               + W_EE @ S_PoS_E
               + W_EI @ S_PoS_I
               + w_ATN_to_PoS_match * S_ATN_E)

    V_PoS_I = (gamma_I
               + W_IE @ S_PoS_E
               + W_II @ S_PoS_I)

    F_PoS_E = (1 + np.tanh(V_PoS_E)) / 2
    F_PoS_I = (1 + np.tanh(V_PoS_I)) / 2

    PoS_E.V[:] = V_PoS_E
    PoS_E.F[:] = F_PoS_E
    PoS_I.V[:] = V_PoS_I
    PoS_I.F[:] = F_PoS_I

    # --- ATN voltage ---
    V_ATN_E = (gamma_E + gain_control
               + W_EE @ S_ATN_E
               + W_EI @ S_ATN_I
               + w_PoS_to_ATN_match * S_PoS_E
               + offset_input)

    V_ATN_I = (gamma_I
               + W_IE @ S_ATN_E
               + W_II @ S_ATN_I)

    F_ATN_E = (1 + np.tanh(V_ATN_E)) / 2
    F_ATN_I = (1 + np.tanh(V_ATN_I)) / 2

    ATN_E.V[:] = V_ATN_E
    ATN_E.F[:] = F_ATN_E
    ATN_I.V[:] = V_ATN_I
    ATN_I.F[:] = F_ATN_I

# ---- Monitors ----
mon_PoS_E = StateMonitor(PoS_E, ['S', 'F'], record=True)
mon_ATN_E = StateMonitor(ATN_E, ['S', 'F'], record=True)

# ---- Run ----
net = Network(PoS_E, PoS_I, ATN_E, ATN_I, update_all,
              mon_PoS_E, mon_ATN_E)

print("Running full model with time-varying velocity (500 ms)...")
net.run(total_duration * ms, report='text')
print("Done!")

In [ ]:
#============================================================
# Static plots of the time-varying velocity simulation
#============================================================

# Precompute bump positions for entire simulation
angle_rad = np.radians(phi_E)
bump_pos_PoS = []
bump_pos_ATN = []

for t_idx in range(len(mon_PoS_E.t)):
    F_pos = mon_PoS_E.F[:, t_idx]
    F_atn = mon_ATN_E.F[:, t_idx]

    d_pos, _ = population_vector_readout(F_pos, phi_E)
    d_atn, _ = population_vector_readout(F_atn, phi_E)

    bump_pos_PoS.append(d_pos)
    bump_pos_ATN.append(d_atn)

bump_pos_PoS = np.array(bump_pos_PoS)
bump_pos_ATN = np.array(bump_pos_ATN)
time_ms = np.array(mon_PoS_E.t / ms)

fig, axes = plt.subplots(3, 1, figsize=(14, 12),
                          gridspec_kw={'height_ratios': [1, 2, 2]})

# --- Top: velocity input ---
axes[0].plot(time_points, velocity_signal, 'k-', linewidth=2)
axes[0].fill_between(time_points, velocity_signal, 0,
                     where=velocity_signal > 0, alpha=0.3, color='blue')
axes[0].fill_between(time_points, velocity_signal, 0,
                     where=velocity_signal < 0, alpha=0.3, color='red')
axes[0].axhline(y=0, color='gray', linewidth=0.5)
axes[0].set_ylabel('Angular vel (°/s)')
axes[0].set_title('Input: Angular Velocity Signal', fontsize=13)
axes[0].set_xlim(0, total_duration)

# --- Middle: heatmap of PoS:E activity ---
axes[1].imshow(mon_PoS_E.F[:], aspect='auto', cmap='hot',
               extent=[0, total_duration, 360, 0], vmin=0, vmax=1)
axes[1].set_ylabel('Preferred direction (°)')
axes[1].set_title('PoS:E Activity — Watch the bump move!', fontsize=13)

# --- Bottom: bump position over time ---
axes[2].plot(time_ms, bump_pos_PoS, 'r-', linewidth=2, label='PoS:E')
axes[2].plot(time_ms, bump_pos_ATN, 'b--', linewidth=1.5, label='ATN:E')
axes[2].set_xlabel('Time (ms)')
axes[2].set_ylabel('Represented direction (°)')
axes[2].set_title('Decoded Head Direction Over Time', fontsize=13)
axes[2].legend(fontsize=11)
axes[2].set_xlim(0, total_duration)

# Add phase labels
phase_labels = ['Right\n150°/s', 'Stop', 'Left\n-200°/s', 'Right\n400°/s', 'Stop']
phase_starts = [0, 100, 150, 300, 400]
phase_ends = [100, 150, 300, 400, 500]
phase_colors = ['blue', 'gray', 'red', 'blue', 'gray']

for start, end, label, color in zip(phase_starts, phase_ends, phase_labels, phase_colors):
    for ax in axes:
        ax.axvspan(start, end, alpha=0.05, color=color)
    axes[2].text((start + end) / 2, axes[2].get_ylim()[1] * 0.95, label,
                ha='center', va='top', fontsize=9, color=color,
                fontweight='bold')

plt.suptitle('Step 9: Full Model with Time-Varying Velocity Input',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#============================================================
# ANIMATION — Watch the bump respond to changing velocity
#============================================================
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

total_steps = len(mon_PoS_E.t)
frame_skip = max(1, total_steps // 300)
n_frames = total_steps // frame_skip

# ---- Top-left: bump profile ----
line_pos, = axes[0, 0].plot(phi_E, mon_PoS_E.F[:, 0], 'r-', linewidth=2, label='PoS:E')
line_atn, = axes[0, 0].plot(phi_E, mon_ATN_E.F[:, 0], 'b--', linewidth=2, label='ATN:E')
axes[0, 0].set_xlim(0, 360)
axes[0, 0].set_ylim(-0.05, 1.05)
axes[0, 0].set_xlabel('Preferred direction (°)')
axes[0, 0].set_ylabel('Firing rate')
axes[0, 0].legend(loc='upper right')
axes[0, 0].set_title('Bump Profile')
time_text = axes[0, 0].text(0.02, 0.92, '', transform=axes[0, 0].transAxes,
                             fontsize=12, fontweight='bold', va='top',
                             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# ---- Top-right: circular representation ----
theta_circle = np.linspace(0, 2*np.pi, 100)
axes[0, 1].plot(np.cos(theta_circle), np.sin(theta_circle), 'k-', linewidth=1)
bump_dot_pos, = axes[0, 1].plot([], [], 'ro', markersize=15, label='PoS', zorder=5)
bump_dot_atn, = axes[0, 1].plot([], [], 'bs', markersize=12, label='ATN', zorder=5)
bump_arc_pos, = axes[0, 1].plot([], [], 'r-', linewidth=3, alpha=0.5)
axes[0, 1].set_xlim(-1.5, 1.5)
axes[0, 1].set_ylim(-1.5, 1.5)
axes[0, 1].set_aspect('equal')
axes[0, 1].legend(loc='upper left', fontsize=10)
axes[0, 1].set_title('Head Direction on Circle')
# Add degree markers
for angle, label in [(0, '0°'), (90, '90°'), (180, '180°'), (270, '270°')]:
    rad = np.radians(90 - angle)
    axes[0, 1].text(1.2*np.cos(rad), 1.2*np.sin(rad), label,
                    ha='center', va='center', fontsize=10, color='gray')
vel_text = axes[0, 1].text(0, 0, '', ha='center', va='center', fontsize=11,
                            fontweight='bold')

# ---- Bottom-left: velocity input ----
axes[1, 0].plot(time_points, velocity_signal, 'k-', linewidth=1.5)
axes[1, 0].fill_between(time_points, velocity_signal, 0,
                        where=velocity_signal > 0, alpha=0.2, color='blue')
axes[1, 0].fill_between(time_points, velocity_signal, 0,
                        where=velocity_signal < 0, alpha=0.2, color='red')
axes[1, 0].axhline(y=0, color='gray', linewidth=0.5)
vel_marker, = axes[1, 0].plot([], [], 'ko', markersize=8, zorder=5)
vel_vline = axes[1, 0].axvline(x=0, color='green', linewidth=2, alpha=0.7)
axes[1, 0].set_xlabel('Time (ms)')
axes[1, 0].set_ylabel('Angular velocity (°/s)')
axes[1, 0].set_title('Input Signal')
axes[1, 0].set_xlim(0, total_duration)

# ---- Bottom-right: bump position trail ----
trail_pos, = axes[1, 1].plot([], [], 'r-', linewidth=2, label='PoS:E')
trail_atn, = axes[1, 1].plot([], [], 'b--', linewidth=1.5, label='ATN:E')
axes[1, 1].set_xlim(0, total_duration)
axes[1, 1].set_ylim(0, 360)
axes[1, 1].set_xlabel('Time (ms)')
axes[1, 1].set_ylabel('Direction (°)')
axes[1, 1].set_title('Bump Position Over Time')
axes[1, 1].legend(fontsize=10)

plt.suptitle('Real-Time Animation — Head Direction System',
             fontsize=15, fontweight='bold')
plt.tight_layout()

def animate(frame):
    t_idx = frame * frame_skip
    if t_idx >= total_steps:
        t_idx = total_steps - 1

    t_ms = time_ms[t_idx]

    # Update bump profiles
    line_pos.set_ydata(mon_PoS_E.F[:, t_idx])
    line_atn.set_ydata(mon_ATN_E.F[:, t_idx])

    # Update time and velocity text
    vel_idx = min(int(t_ms / float(dt_sim/ms)), len(velocity_signal) - 1)
    vel_idx = min(vel_idx, len(velocity_signal) - 1)
    current_vel = velocity_signal[min(t_idx, len(velocity_signal)-1)]

    direction_word = "RIGHT →" if current_vel > 0 else "← LEFT" if current_vel < 0 else "STOPPED"
    time_text.set_text(f't = {t_ms:.0f} ms\n{direction_word}')

    # Update circular representation
    pos_dir = bump_pos_PoS[t_idx]
    atn_dir = bump_pos_ATN[t_idx]
    pos_rad = np.radians(90 - pos_dir)
    atn_rad = np.radians(90 - atn_dir)

    bump_dot_pos.set_data([np.cos(pos_rad)], [np.sin(pos_rad)])
    bump_dot_atn.set_data([np.cos(atn_rad)], [np.sin(atn_rad)])

    # Velocity text in circle center
    vel_text.set_text(f'{current_vel:+.0f}°/s')
    vel_text.set_color('blue' if current_vel > 0 else 'red' if current_vel < 0 else 'gray')

    # Update velocity marker
    vel_marker.set_data([t_ms], [current_vel])
    vel_vline.set_xdata([t_ms])

    # Update trails
    trail_pos.set_data(time_ms[:t_idx], bump_pos_PoS[:t_idx])
    trail_atn.set_data(time_ms[:t_idx], bump_pos_ATN[:t_idx])

    return (line_pos, line_atn, time_text, bump_dot_pos, bump_dot_atn,
            vel_text, vel_marker, vel_vline, trail_pos, trail_atn, bump_arc_pos)

anim = FuncAnimation(fig, animate, frames=n_frames, interval=40, blit=True)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
#============================================================
# Mount Google Drive and load Professor's data
#============================================================
import pickle
from google.colab import drive

# This will open a popup asking you to sign in to your Google account
drive.mount('/content/drive')

# ============================================================
# CHANGE THIS PATH to match where your files are in Google Drive
# For example, if your files are in a folder called "internship_data":
# data_path = '/content/drive/MyDrive/internship_data/export.pkl'
# ============================================================
data_path = '/content/drive/MyDrive/HD_model_extract/export.pkl'

with open(data_path, 'rb') as fp:
    times, traj, vel, acc = pickle.load(fp)
    units, structs, struct_of_unit, units_by_struct = pickle.load(fp)
    spike_times_of_unit, spike_angles_of_unit = pickle.load(fp)

DEG = np.pi / 180

print("Data loaded successfully!")
print(f"\n--- Recording summary ---")
print(f"Duration: {times[-1] - times[0]:.0f} seconds ({(times[-1]-times[0])/60:.1f} minutes)")
print(f"Time samples: {len(times)} (sampled at {1/np.median(np.diff(times)):.0f} Hz)")
print(f"Total neurons: {len(units)}")
print(f"Total spikes: {sum(len(spike_times_of_unit[u]) for u in units)}")

print(f"\n--- Brain structures ---")
for s in structs:
    print(f"  {s}: {len(units_by_struct[s])} units")

In [ ]:
#============================================================
# Visualize the head direction trajectory and angular velocity
#============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10),
                          gridspec_kw={'height_ratios': [2, 1, 1]})

# --- Head direction over time ---
traj_deg = np.degrees(traj) % 360
axes[0].plot(times, traj_deg, 'k-', linewidth=0.5, alpha=0.7)
axes[0].set_ylabel('Head direction (°)')
axes[0].set_title('Animal Head Direction Over Time', fontsize=14)
axes[0].set_ylim(0, 360)
axes[0].set_yticks([0, 90, 180, 270, 360])

# --- Angular velocity ---
vel_deg = np.degrees(vel)
axes[1].plot(times, vel_deg, 'b-', linewidth=0.5, alpha=0.7)
axes[1].axhline(y=0, color='gray', linewidth=0.5)
axes[1].set_ylabel('Angular velocity (°/s)')
axes[1].set_title('Angular Velocity', fontsize=14)

# --- Velocity histogram ---
axes[2].hist(vel_deg, bins=100, color='steelblue', edgecolor='none', alpha=0.7)
axes[2].axvline(x=0, color='gray', linewidth=0.5)
axes[2].set_xlabel('Angular velocity (°/s)')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Velocity Distribution (range: {vel_deg.min():.0f}°/s to {vel_deg.max():.0f}°/s)',
                   fontsize=14)

plt.suptitle("Professor's Recording — Head Direction Data",
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#============================================================
# Find the best HD cells and plot their tuning curves + spike rasters
#============================================================

def rayleigh_vector_length(spike_angles):
    """Measure of directional tuning strength (0 = no tuning, 1 = perfect)"""
    angles = spike_angles % (2 * np.pi)
    if len(angles) == 0:
        return 0, 0
    r = np.abs(np.mean(np.exp(1j * angles)))
    preferred = np.angle(np.mean(np.exp(1j * angles))) % (2 * np.pi)
    return r, preferred

# Score all units in HD-related structures
hd_candidates = []
for struct in ['presubi', 'POST', 'PARA']:
    for u in units_by_struct[struct]:
        angles = spike_angles_of_unit[u]
        if len(angles) < 50:
            continue
        r, pref = rayleigh_vector_length(angles)
        rate = len(angles) / 300.0
        hd_candidates.append({
            'unit': u, 'struct': struct, 'n_spikes': len(angles),
            'rate': rate, 'rayleigh': r, 'pref_deg': np.degrees(pref)
        })

# Sort by tuning strength
hd_candidates.sort(key=lambda x: x['rayleigh'], reverse=True)

print(f"Found {len(hd_candidates)} HD cell candidates")
print(f"\nTop 10 by directional tuning strength:")
print(f"{'Unit':>6} {'Structure':>10} {'Spikes':>8} {'Rate(Hz)':>10} {'Rayleigh':>10} {'Pref dir':>10}")
print("-" * 60)
for c in hd_candidates[:10]:
    print(f"{c['unit']:>6} {c['struct']:>10} {c['n_spikes']:>8} "
          f"{c['rate']:>10.1f} {c['rayleigh']:>10.3f} {c['pref_deg']:>9.1f}°")

In [ ]:
#============================================================
# Plot tuning curves and spike rasters for top 6 HD cells
#============================================================

top_cells = hd_candidates[:6]

fig, axes = plt.subplots(3, 6, figsize=(20, 12),
                          gridspec_kw={'height_ratios': [1, 1, 2]})

for col, cell in enumerate(top_cells):
    u = cell['unit']
    spike_t = spike_times_of_unit[u]
    spike_ang = spike_angles_of_unit[u]
    spike_ang_deg = np.degrees(spike_ang) % 360

    # --- Row 1: Tuning curve (histogram of spike angles) ---
    n_bins = 36  # 10° bins
    bins = np.linspace(0, 360, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    # Count spikes per bin
    counts, _ = np.histogram(spike_ang_deg, bins=bins)

    # Convert to firing rate: need time spent in each direction bin
    traj_deg_wrapped = np.degrees(traj) % 360
    dt_sample = np.median(np.diff(times))
    occupancy, _ = np.histogram(traj_deg_wrapped, bins=bins)
    occupancy_time = occupancy * dt_sample  # seconds spent in each bin

    # Firing rate = spikes / time
    with np.errstate(divide='ignore', invalid='ignore'):
        firing_rate = np.where(occupancy_time > 0, counts / occupancy_time, 0)

    axes[0, col].bar(bin_centers, firing_rate, width=9, color='steelblue',
                      edgecolor='none', alpha=0.8)
    axes[0, col].axvline(x=cell['pref_deg'], color='red', linewidth=2,
                          linestyle='--', alpha=0.7)
    axes[0, col].set_title(f"Unit {u} ({cell['struct']})\n"
                           f"R={cell['rayleigh']:.2f}, pref={cell['pref_deg']:.0f}°",
                           fontsize=10)
    if col == 0:
        axes[0, col].set_ylabel('Firing rate (Hz)')
    axes[0, col].set_xlim(0, 360)
    axes[0, col].set_xticks([0, 90, 180, 270, 360])

    # --- Row 2: Polar tuning curve ---
    theta_bins = np.radians(bin_centers)
    # Close the polar plot
    theta_polar = np.append(theta_bins, theta_bins[0])
    rate_polar = np.append(firing_rate, firing_rate[0])

    # Remove the rectangular axis and create polar
    axes[1, col].remove()
    ax_polar = fig.add_subplot(3, 6, 7 + col, projection='polar')
    ax_polar.plot(theta_polar, rate_polar, 'steelblue', linewidth=2)
    ax_polar.fill(theta_polar, rate_polar, 'steelblue', alpha=0.3)
    ax_polar.set_title(f'Polar tuning', fontsize=9)
    pref_rad = np.radians(cell['pref_deg'])
    ax_polar.plot([pref_rad, pref_rad], [0, max(firing_rate)], 'r--', linewidth=2)

    # --- Row 3: Spike raster (spikes plotted as dots on the trajectory) ---
    # Show first 60 seconds for clarity
    t_max = 60
    mask_traj = times <= t_max
    mask_spikes = spike_t <= t_max

    axes[2, col].plot(times[mask_traj], traj_deg_wrapped[mask_traj],
                      'k-', linewidth=0.5, alpha=0.3, label='Head dir')
    axes[2, col].scatter(spike_t[mask_spikes], spike_ang_deg[mask_spikes],
                         s=3, c='red', alpha=0.5, label='Spikes', zorder=3)
    axes[2, col].set_ylim(0, 360)
    axes[2, col].set_yticks([0, 90, 180, 270, 360])
    axes[2, col].set_xlim(0, t_max)
    if col == 0:
        axes[2, col].set_ylabel('Direction (°)')
        axes[2, col].legend(fontsize=8, loc='upper right')
    axes[2, col].set_xlabel('Time (s)')

plt.suptitle("Real Head Direction Cells — Tuning Curves & Spike Rasters",
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#============================================================
# Show all HD cells' preferred directions on a circle
#============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: all HD cells on a circle, colored by structure ---
theta_circle = np.linspace(0, 2*np.pi, 360)
axes[0].plot(np.cos(theta_circle), np.sin(theta_circle), 'k-', linewidth=1)

colors_struct = {'presubi': 'red', 'POST': 'blue', 'PARA': 'green'}
for cell in hd_candidates:
    if cell['rayleigh'] > 0.5:  # only show well-tuned cells
        angle_rad = np.radians(90 - cell['pref_deg'])
        r = 0.6 + 0.4 * cell['rayleigh']  # distance from center = tuning strength
        x = r * np.cos(angle_rad)
        y = r * np.sin(angle_rad)
        axes[0].scatter(x, y, c=colors_struct[cell['struct']],
                       s=30 + cell['n_spikes']/10, alpha=0.6, edgecolors='k', linewidth=0.5)

# Add legend
for struct, color in colors_struct.items():
    n = sum(1 for c in hd_candidates if c['struct'] == struct and c['rayleigh'] > 0.5)
    axes[0].scatter([], [], c=color, s=50, label=f'{struct} ({n} cells)', edgecolors='k')
axes[0].legend(fontsize=10, loc='lower left')

for angle, label in [(0, '0°'), (90, '90°'), (180, '180°'), (270, '270°')]:
    rad = np.radians(90 - angle)
    axes[0].text(1.15*np.cos(rad), 1.15*np.sin(rad), label,
                ha='center', va='center', fontsize=11, color='gray')

axes[0].set_xlim(-1.4, 1.4)
axes[0].set_ylim(-1.4, 1.4)
axes[0].set_aspect('equal')
axes[0].axis('off')
axes[0].set_title('HD Cell Preferred Directions\n(distance from center = tuning strength)',
                   fontsize=13)

# --- Right: distribution of preferred directions ---
pref_dirs = [c['pref_deg'] for c in hd_candidates if c['rayleigh'] > 0.5]
axes[1].hist(pref_dirs, bins=36, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Preferred direction (°)')
axes[1].set_ylabel('Number of cells')
axes[1].set_title('Distribution of Preferred Directions\n(cells with Rayleigh > 0.5)',
                   fontsize=13)
axes[1].set_xlim(0, 360)
axes[1].set_xticks([0, 90, 180, 270, 360])

plt.suptitle("Head Direction Cell Population — Professor's Data",
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

n_good = sum(1 for c in hd_candidates if c['rayleigh'] > 0.5)
print(f"\nFound {n_good} well-tuned HD cells (Rayleigh > 0.5)")
print(f"Preferred directions span the full 360° — just as the Redish model assumes")

In [ ]:
#============================================================
# STEP 10: Prepare real angular velocity data for the model
#============================================================

# Select a 20-second segment of the recording
t_start = 50.0   # start time in seconds (skip the first 50s)
t_end = 70.0     # end time in seconds
segment_mask = (times >= t_start) & (times <= t_end)

times_segment = times[segment_mask]
traj_segment = traj[segment_mask]       # radians
vel_segment = vel[segment_mask]         # radians/s

# Convert to degrees for our model
traj_segment_deg = np.degrees(traj_segment) % 360
vel_segment_deg = np.degrees(vel_segment)   # degrees/s

# The recording is sampled at 10 Hz (every 100 ms)
# Our model runs at 0.1 ms timestep
# We need to interpolate velocity to the model's timestep

model_dt = 0.1e-3  # 0.1 ms in seconds
model_times = np.arange(times_segment[0], times_segment[-1], model_dt)

# Interpolate velocity to model timestep
vel_interp_deg = np.interp(model_times, times_segment, vel_segment_deg)

# Also interpolate the true head direction for comparison
traj_interp_deg = np.interp(model_times, times_segment,
                             np.unwrap(traj_segment) * 180 / np.pi)
traj_interp_deg_wrapped = traj_interp_deg % 360

print(f"Segment: {t_start:.0f}s to {t_end:.0f}s ({t_end-t_start:.0f} seconds)")
print(f"Recording samples: {sum(segment_mask)}")
print(f"Model timesteps: {len(model_times)} ({len(model_times)*0.1/1000:.1f} seconds)")
print(f"Velocity range: {vel_interp_deg.min():.1f}°/s to {vel_interp_deg.max():.1f}°/s")

# Visualize the segment we'll simulate
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(times_segment - t_start, traj_segment_deg, 'k-', linewidth=1.5)
axes[0].set_ylabel('Head direction (°)')
axes[0].set_title(f'Selected Segment ({t_start:.0f}s – {t_end:.0f}s)', fontsize=14)
axes[0].set_ylim(0, 360)
axes[0].set_yticks([0, 90, 180, 270, 360])

axes[1].plot(times_segment - t_start, vel_segment_deg, 'b-', linewidth=1)
axes[1].axhline(y=0, color='gray', linewidth=0.5)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Angular velocity (°/s)')
axes[1].set_title('Angular Velocity — This is what drives our model', fontsize=14)

plt.suptitle("Real Data Segment — Input to Model", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#============================================================
# Run the model driven by REAL angular velocity
#============================================================

start_scope()
defaultclock.dt = dt_sim

# ---- Neuron equations ----
eqs_E = '''
dS/dt = (-S + F) / tau_E_param : 1
F : 1
V : 1
'''

eqs_I = '''
dS/dt = (-S + F) / tau_I_param : 1
F : 1
V : 1
'''

# ---- Create all four pools ----
PoS_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='PoS_E')
PoS_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='PoS_I')
ATN_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='ATN_E')
ATN_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='ATN_I')

# ---- Offset index mappings ----
offset_steps = int(round(delta_offset / (360.0 / N_E)))
right_offset_indices = np.array([(i + offset_steps) % N_E for i in range(N_E)])
left_offset_indices = np.array([(i - offset_steps) % N_E for i in range(N_E)])

# ---- Initialize bumps at the ACTUAL initial head direction ----
initial_hd = traj_interp_deg_wrapped[0]
print(f"Initializing bump at actual head direction: {initial_hd:.1f}°")

for i in range(N_E):
    angle_diff = min(abs(phi_E[i] - initial_hd), 360 - abs(phi_E[i] - initial_hd))
    bump_val = 0.5 * np.exp(-angle_diff**2 / 30**2)
    PoS_E.S[i] = bump_val
    ATN_E.S[i] = bump_val

PoS_I.S = np.random.uniform(0, 0.01, N_I)
ATN_I.S = np.random.uniform(0, 0.01, N_I)

# ---- Step counter and velocity array ----
current_step = [0]

# We need to recalibrate xi so bump speed matches real velocity
# From Step 8 we saw the bump moves ~3x too fast
# Adjust by scaling the xi function
def xi_calibrated(angular_velocity):
    v = np.abs(angular_velocity)
    return 0.33 * v / (v + 200)  # scale down by ~3x

# ---- Network operation ----
@network_operation(dt=defaultclock.dt)
def update_all():
    S_PoS_E = np.array(PoS_E.S[:])
    S_PoS_I = np.array(PoS_I.S[:])
    S_ATN_E = np.array(ATN_E.S[:])
    S_ATN_I = np.array(ATN_I.S[:])

    step = current_step[0]
    if step < len(vel_interp_deg):
        ang_vel = vel_interp_deg[step]
    else:
        ang_vel = 0.0
    current_step[0] += 1

    xi_val = xi_calibrated(ang_vel)
    gain_control = -0.5 * xi_val

    if ang_vel > 0:
        offset_input = xi_val * S_PoS_E[left_offset_indices]
    elif ang_vel < 0:
        offset_input = xi_val * S_PoS_E[right_offset_indices]
    else:
        offset_input = np.zeros(N_E)

    V_PoS_E = (gamma_E
               + W_EE @ S_PoS_E
               + W_EI @ S_PoS_I
               + w_ATN_to_PoS_match * S_ATN_E)

    V_PoS_I = (gamma_I
               + W_IE @ S_PoS_E
               + W_II @ S_PoS_I)

    F_PoS_E = (1 + np.tanh(V_PoS_E)) / 2
    F_PoS_I = (1 + np.tanh(V_PoS_I)) / 2

    PoS_E.V[:] = V_PoS_E
    PoS_E.F[:] = F_PoS_E
    PoS_I.V[:] = V_PoS_I
    PoS_I.F[:] = F_PoS_I

    V_ATN_E = (gamma_E + gain_control
               + W_EE @ S_ATN_E
               + W_EI @ S_ATN_I
               + w_PoS_to_ATN_match * S_PoS_E
               + offset_input)

    V_ATN_I = (gamma_I
               + W_IE @ S_ATN_E
               + W_II @ S_ATN_I)

    F_ATN_E = (1 + np.tanh(V_ATN_E)) / 2
    F_ATN_I = (1 + np.tanh(V_ATN_I)) / 2

    ATN_E.V[:] = V_ATN_E
    ATN_E.F[:] = F_ATN_E
    ATN_I.V[:] = V_ATN_I
    ATN_I.F[:] = F_ATN_I

# ---- Monitor: only record every 1ms to save memory ----
mon_PoS_E = StateMonitor(PoS_E, 'F', record=True, dt=1*ms)
mon_ATN_E = StateMonitor(ATN_E, 'F', record=True, dt=1*ms)

# ---- Run ----
net = Network(PoS_E, PoS_I, ATN_E, ATN_I, update_all,
              mon_PoS_E, mon_ATN_E)

sim_duration = t_end - t_start
print(f"Running model for {sim_duration:.0f} seconds with REAL angular velocity...")
print(f"This may take a few minutes...")
net.run(sim_duration * second, report='text')
print("Done!")

In [ ]:
#============================================================
# Compare model output to real head direction (Figure 5 from paper)
#============================================================

# Compute bump position at each recorded timestep
model_time_s = np.array(mon_PoS_E.t / second)
bump_pos_model = []

for t_idx in range(len(mon_PoS_E.t)):
    F_pos = mon_PoS_E.F[:, t_idx]
    d, _ = population_vector_readout(F_pos, phi_E)
    bump_pos_model.append(d)

bump_pos_model = np.array(bump_pos_model)

# Interpolate true head direction to model monitor times
true_hd_at_monitor = np.interp(model_time_s + t_start, times,
                                np.unwrap(traj) * 180 / np.pi)
true_hd_wrapped = true_hd_at_monitor % 360

# --- Compute tracking error ---
error = bump_pos_model - true_hd_wrapped
# Handle wraparound
error = (error + 180) % 360 - 180

fig, axes = plt.subplots(3, 1, figsize=(14, 12),
                          gridspec_kw={'height_ratios': [3, 1, 1]})

# --- Top: model vs real head direction (Figure 5 equivalent) ---
axes[0].plot(model_time_s, true_hd_wrapped, 'k-', linewidth=1.5,
             label='Real head direction', alpha=0.8)
axes[0].plot(model_time_s, bump_pos_model, 'r--', linewidth=1.5,
             label='Model (PoS:E)', alpha=0.8)
axes[0].set_ylabel('Direction (°)', fontsize=12)
axes[0].set_title('Model Tracking Real Head Direction (cf. Figure 5 in Redish et al.)',
                   fontsize=14)
axes[0].legend(fontsize=12)
axes[0].set_ylim(0, 360)
axes[0].set_yticks([0, 90, 180, 270, 360])

# --- Middle: tracking error (Figure 6 equivalent) ---
axes[1].plot(model_time_s, error, 'b-', linewidth=1, alpha=0.7)
axes[1].axhline(y=0, color='gray', linewidth=0.5)
axes[1].set_ylabel('Error (°)', fontsize=12)
axes[1].set_title('Tracking Error (cf. Figure 6 in Redish et al.)', fontsize=14)
axes[1].set_ylim(-90, 90)

# --- Bottom: angular velocity input ---
vel_at_monitor = np.interp(model_time_s + t_start, times, np.degrees(vel))
axes[2].plot(model_time_s, vel_at_monitor, 'g-', linewidth=0.8, alpha=0.7)
axes[2].axhline(y=0, color='gray', linewidth=0.5)
axes[2].set_xlabel('Time (s)', fontsize=12)
axes[2].set_ylabel('Angular vel (°/s)', fontsize=12)
axes[2].set_title('Input Angular Velocity', fontsize=14)

plt.suptitle("Step 10: Model vs Real Recording — Does our model track real head direction?",
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
print(f"\n--- Tracking Performance ---")
print(f"Mean absolute error: {np.mean(np.abs(error)):.1f}°")
print(f"Max absolute error: {np.max(np.abs(error)):.1f}°")
print(f"RMS error: {np.sqrt(np.mean(error**2)):.1f}°")
print(f"\nRedish et al. report error typically < 20° for simulations < 3 min")

In [ ]:
#============================================================
# Overlay REAL spikes on the model's activity
#============================================================

# Pick the top 4 HD cells from the data
top4 = hd_candidates[:4]

fig, axes = plt.subplots(len(top4) + 1, 1, figsize=(14, 3 * (len(top4) + 1)),
                          gridspec_kw={'height_ratios': [2] + [1]*len(top4)})

# --- Top panel: model bump activity with real HD overlaid ---
axes[0].imshow(mon_PoS_E.F[:], aspect='auto', cmap='hot',
               extent=[0, model_time_s[-1], 360, 0], vmin=0, vmax=1)
axes[0].plot(model_time_s, true_hd_wrapped, 'c-', linewidth=1, alpha=0.7,
             label='Real head dir')
axes[0].set_ylabel('Direction (°)')
axes[0].set_title('Model PoS:E Activity + Real Head Direction', fontsize=14)
axes[0].legend(loc='upper right', fontsize=10)
axes[0].set_ylim(360, 0)

# --- One panel per real HD cell ---
for idx, cell in enumerate(top4):
    u = cell['unit']
    spike_t = spike_times_of_unit[u]
    spike_ang = np.degrees(spike_angles_of_unit[u]) % 360

    # Filter spikes to our time segment
    mask = (spike_t >= t_start) & (spike_t <= t_end)
    spike_t_seg = spike_t[mask] - t_start
    spike_ang_seg = spike_ang[mask]

    # Plot trajectory
    axes[idx + 1].plot(model_time_s, true_hd_wrapped, 'k-', linewidth=0.5, alpha=0.3)

    # Plot spikes
    axes[idx + 1].scatter(spike_t_seg, spike_ang_seg, s=8, c='red',
                          alpha=0.7, zorder=3)

    # Mark preferred direction
    axes[idx + 1].axhline(y=cell['pref_deg'], color='blue', linewidth=1,
                          linestyle='--', alpha=0.5)

    axes[idx + 1].set_ylabel('Direction (°)')
    axes[idx + 1].set_title(f"Unit {u} ({cell['struct']}) — pref={cell['pref_deg']:.0f}°, "
                            f"R={cell['rayleigh']:.2f}", fontsize=11)
    axes[idx + 1].set_ylim(0, 360)
    axes[idx + 1].set_yticks([0, 90, 180, 270, 360])

axes[-1].set_xlabel('Time (s)')

plt.suptitle("Real Spikes vs Model Activity — Same Time Segment",
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each red dot is a REAL spike from the animal's brain.")
print("Spikes cluster near each neuron's preferred direction (blue dashed line).")
print("The model's bump (top panel) should track through these directions")
print("at the same times the real neurons fire.")

In [ ]:
#============================================================
# Recalibrate ξ function by testing different scaling factors
#============================================================

# Instead of rerunning the full simulation many times (slow),
# we'll do a quick analytical estimate.
#
# The key insight: we know the REAL velocity and the REAL direction.
# We can measure how much the bump SHOULD move per timestep
# and compare to how much it DOES move.
#
# Let's test a few scaling factors on a shorter segment first.

from brian2 import *
import time as time_module

def run_short_test(xi_scale, test_duration=5.0):
    """Run a short simulation with a given xi scaling factor"""
    start_scope()
    defaultclock.dt = dt_sim

    eqs_E = '''
    dS/dt = (-S + F) / tau_E_param : 1
    F : 1
    V : 1
    '''
    eqs_I = '''
    dS/dt = (-S + F) / tau_I_param : 1
    F : 1
    V : 1
    '''

    PoS_E_t = NeuronGroup(N_E, eqs_E, method='euler',
                          namespace={'tau_E_param': tau_E}, name='PoS_E')
    PoS_I_t = NeuronGroup(N_I, eqs_I, method='euler',
                          namespace={'tau_I_param': tau_I}, name='PoS_I')
    ATN_E_t = NeuronGroup(N_E, eqs_E, method='euler',
                          namespace={'tau_E_param': tau_E}, name='ATN_E')
    ATN_I_t = NeuronGroup(N_I, eqs_I, method='euler',
                          namespace={'tau_I_param': tau_I}, name='ATN_I')

    o_steps = int(round(delta_offset / (360.0 / N_E)))
    r_idx = np.array([(i + o_steps) % N_E for i in range(N_E)])
    l_idx = np.array([(i - o_steps) % N_E for i in range(N_E)])

    init_hd = traj_interp_deg_wrapped[0]
    for i in range(N_E):
        ad = min(abs(phi_E[i] - init_hd), 360 - abs(phi_E[i] - init_hd))
        bv = 0.5 * np.exp(-ad**2 / 30**2)
        PoS_E_t.S[i] = bv
        ATN_E_t.S[i] = bv
    PoS_I_t.S = np.random.uniform(0, 0.01, N_I)
    ATN_I_t.S = np.random.uniform(0, 0.01, N_I)

    step_counter = [0]

    def xi_test(av):
        v = np.abs(av)
        return xi_scale * v / (v + 200)

    @network_operation(dt=defaultclock.dt)
    def update():
        S_pE = np.array(PoS_E_t.S[:])
        S_pI = np.array(PoS_I_t.S[:])
        S_aE = np.array(ATN_E_t.S[:])
        S_aI = np.array(ATN_I_t.S[:])

        s = step_counter[0]
        av = vel_interp_deg[s] if s < len(vel_interp_deg) else 0.0
        step_counter[0] += 1

        xv = xi_test(av)
        gc = -0.5 * xv

        if av > 0:
            oi = xv * S_pE[l_idx]
        elif av < 0:
            oi = xv * S_pE[r_idx]
        else:
            oi = np.zeros(N_E)

        VpE = gamma_E + W_EE @ S_pE + W_EI @ S_pI + w_ATN_to_PoS_match * S_aE
        VpI = gamma_I + W_IE @ S_pE + W_II @ S_pI
        PoS_E_t.V[:] = VpE
        PoS_E_t.F[:] = (1 + np.tanh(VpE)) / 2
        PoS_I_t.V[:] = VpI
        PoS_I_t.F[:] = (1 + np.tanh(VpI)) / 2

        VaE = gamma_E + gc + W_EE @ S_aE + W_EI @ S_aI + w_PoS_to_ATN_match * S_pE + oi
        VaI = gamma_I + W_IE @ S_aE + W_II @ S_aI
        ATN_E_t.V[:] = VaE
        ATN_E_t.F[:] = (1 + np.tanh(VaE)) / 2
        ATN_I_t.V[:] = VaI
        ATN_I_t.F[:] = (1 + np.tanh(VaI)) / 2

    mon = StateMonitor(PoS_E_t, 'F', record=True, dt=1*ms)
    net = Network(PoS_E_t, PoS_I_t, ATN_E_t, ATN_I_t, update, mon)
    net.run(test_duration * second)

    # Compute error
    mt = np.array(mon.t / second)
    bp = []
    for t_idx in range(len(mt)):
        d, _ = population_vector_readout(mon.F[:, t_idx], phi_E)
        bp.append(d)
    bp = np.array(bp)

    true_hd = np.interp(mt + t_start, times, np.unwrap(traj) * 180 / np.pi) % 360
    err = (bp - true_hd + 180) % 360 - 180

    return np.mean(np.abs(err)), np.sqrt(np.mean(err**2))

# Test different scaling factors
print("Testing different ξ scaling factors (5-second test each)...")
print(f"{'Scale':>8} {'Mean error':>12} {'RMS error':>12}")
print("-" * 36)

scales = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
results = []

for scale in scales:
    t0 = time_module.time()
    mae, rmse = run_short_test(scale, test_duration=5.0)
    elapsed = time_module.time() - t0
    results.append((scale, mae, rmse))
    print(f"{scale:>8.2f} {mae:>11.1f}° {rmse:>11.1f}° ({elapsed:.0f}s)")

best = min(results, key=lambda x: x[1])
print(f"\nBest scaling factor: {best[0]:.2f} (mean error = {best[1]:.1f}°)")

**Update ξ and rerun full simulation**

In [ ]:
#============================================================
# Rerun full 20s simulation with BEST ξ scaling factor
#============================================================

start_scope()
defaultclock.dt = dt_sim

# ---- Updated xi function with best scaling ----
def xi_calibrated(angular_velocity):
    v = np.abs(angular_velocity)
    return 0.20 * v / (v + 200)

print(f"Using ξ scaling factor = 0.20 (best from calibration)")

# ---- Neuron equations ----
eqs_E = '''
dS/dt = (-S + F) / tau_E_param : 1
F : 1
V : 1
'''
eqs_I = '''
dS/dt = (-S + F) / tau_I_param : 1
F : 1
V : 1
'''

# ---- Create all four pools ----
PoS_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='PoS_E')
PoS_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='PoS_I')
ATN_E = NeuronGroup(N_E, eqs_E, method='euler',
                     namespace={'tau_E_param': tau_E}, name='ATN_E')
ATN_I = NeuronGroup(N_I, eqs_I, method='euler',
                     namespace={'tau_I_param': tau_I}, name='ATN_I')

# ---- Offset index mappings ----
offset_steps = int(round(delta_offset / (360.0 / N_E)))
right_offset_indices = np.array([(i + offset_steps) % N_E for i in range(N_E)])
left_offset_indices = np.array([(i - offset_steps) % N_E for i in range(N_E)])

# ---- Initialize at actual head direction ----
initial_hd = traj_interp_deg_wrapped[0]
print(f"Initializing bump at: {initial_hd:.1f}°")

for i in range(N_E):
    angle_diff = min(abs(phi_E[i] - initial_hd), 360 - abs(phi_E[i] - initial_hd))
    bump_val = 0.5 * np.exp(-angle_diff**2 / 30**2)
    PoS_E.S[i] = bump_val
    ATN_E.S[i] = bump_val

PoS_I.S = np.random.uniform(0, 0.01, N_I)
ATN_I.S = np.random.uniform(0, 0.01, N_I)

# ---- Step counter ----
current_step = [0]

# ---- Network operation ----
@network_operation(dt=defaultclock.dt)
def update_all():
    S_PoS_E = np.array(PoS_E.S[:])
    S_PoS_I = np.array(PoS_I.S[:])
    S_ATN_E = np.array(ATN_E.S[:])
    S_ATN_I = np.array(ATN_I.S[:])

    step = current_step[0]
    if step < len(vel_interp_deg):
        ang_vel = vel_interp_deg[step]
    else:
        ang_vel = 0.0
    current_step[0] += 1

    xi_val = xi_calibrated(ang_vel)
    gain_control = -0.5 * xi_val

    if ang_vel > 0:
        offset_input = xi_val * S_PoS_E[left_offset_indices]
    elif ang_vel < 0:
        offset_input = xi_val * S_PoS_E[right_offset_indices]
    else:
        offset_input = np.zeros(N_E)

    V_PoS_E = (gamma_E
               + W_EE @ S_PoS_E
               + W_EI @ S_PoS_I
               + w_ATN_to_PoS_match * S_ATN_E)
    V_PoS_I = (gamma_I
               + W_IE @ S_PoS_E
               + W_II @ S_PoS_I)
    F_PoS_E = (1 + np.tanh(V_PoS_E)) / 2
    F_PoS_I = (1 + np.tanh(V_PoS_I)) / 2
    PoS_E.V[:] = V_PoS_E
    PoS_E.F[:] = F_PoS_E
    PoS_I.V[:] = V_PoS_I
    PoS_I.F[:] = F_PoS_I

    V_ATN_E = (gamma_E + gain_control
               + W_EE @ S_ATN_E
               + W_EI @ S_ATN_I
               + w_PoS_to_ATN_match * S_PoS_E
               + offset_input)
    V_ATN_I = (gamma_I
               + W_IE @ S_ATN_E
               + W_II @ S_ATN_I)
    F_ATN_E = (1 + np.tanh(V_ATN_E)) / 2
    F_ATN_I = (1 + np.tanh(V_ATN_I)) / 2
    ATN_E.V[:] = V_ATN_E
    ATN_E.F[:] = F_ATN_E
    ATN_I.V[:] = V_ATN_I
    ATN_I.F[:] = F_ATN_I

# ---- Monitors ----
mon_PoS_E = StateMonitor(PoS_E, 'F', record=True, dt=1*ms)
mon_ATN_E = StateMonitor(ATN_E, 'F', record=True, dt=1*ms)

# ---- Run ----
net = Network(PoS_E, PoS_I, ATN_E, ATN_I, update_all,
              mon_PoS_E, mon_ATN_E)

sim_duration = t_end - t_start
print(f"Running full {sim_duration:.0f}s simulation with calibrated ξ...")
print("This will take ~7 minutes...")
net.run(sim_duration * second, report='text')
print("Done!")

The only difference from Cell 27 is the ξ function: 0.20 * v / (v + 200) instead of 0.33 * v / (v + 200). This means the offset connections push the ATN bump less aggressively, which slows down the bump's movement to better match the real angular velocity.

**SAVE CHECKPOINTS TO DRIVE**

In [ ]:
#============================================================
# SAVE UPDATED CHECKPOINT (run before closing laptop!)
#============================================================
import pickle
import os

checkpoint_dir = '/content/drive/MyDrive/HD_model_extract/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# ---- Save updated Step 10 results with better calibration ----
model_time_s = np.array(mon_PoS_E.t / second)

# Recompute bump positions with new data
bump_pos_model = []
for t_idx in range(len(mon_PoS_E.t)):
    F_pos = mon_PoS_E.F[:, t_idx]
    d, _ = population_vector_readout(F_pos, phi_E)
    bump_pos_model.append(d)
bump_pos_model = np.array(bump_pos_model)

true_hd_at_monitor = np.interp(model_time_s + t_start, times,
                                np.unwrap(traj) * 180 / np.pi)
true_hd_wrapped = true_hd_at_monitor % 360
error = (bump_pos_model - true_hd_wrapped + 180) % 360 - 180

with open(f'{checkpoint_dir}/step10_results.pkl', 'wb') as f:
    pickle.dump({
        'model_time_s': model_time_s,
        'PoS_F': np.array(mon_PoS_E.F[:]),
        'ATN_F': np.array(mon_ATN_E.F[:]),
        'bump_pos_model': bump_pos_model,
        'true_hd_wrapped': true_hd_wrapped,
        'error': error,
        't_start': t_start,
        't_end': t_end,
        'vel_interp_deg': vel_interp_deg,
        'traj_interp_deg_wrapped': traj_interp_deg_wrapped,
        'model_times': model_times,
        'xi_scale': 0.20,  # save the calibration!
    }, f)
print("✓ Updated Step 10 results saved (with ξ scale = 0.20)")

# ---- Save calibration result ----
with open(f'{checkpoint_dir}/calibration.pkl', 'wb') as f:
    pickle.dump({
        'best_scale': 0.20,
        'mean_error': 19.5,
        'rms_error': 21.4,
        'max_error': 29.7,
    }, f)
print("✓ Calibration result saved")

# ---- Resave everything else ----
with open(f'{checkpoint_dir}/weight_matrices.pkl', 'wb') as f:
    pickle.dump({
        'W_EE': W_EE, 'W_IE': W_IE,
        'W_EI': W_EI, 'W_II': W_II,
        'phi_E': phi_E, 'phi_I': phi_I,
    }, f)
print("✓ Weight matrices saved")

with open(f'{checkpoint_dir}/model_params.pkl', 'wb') as f:
    pickle.dump({
        'N_E': N_E, 'N_I': N_I,
        'tau_E': float(tau_E), 'tau_I': float(tau_I),
        'gamma_E': gamma_E, 'gamma_I': gamma_I,
        'sigma_E': sigma_E, 'sigma_I': sigma_I,
        'kappa_EE': kappa_EE, 'kappa_IE': kappa_IE,
        'kappa_II': kappa_II, 'kappa_EI': kappa_EI,
        'w_PoS_to_ATN_match': w_PoS_to_ATN_match,
        'w_ATN_to_PoS_match': w_ATN_to_PoS_match,
        'delta_offset': delta_offset,
    }, f)
print("✓ Model parameters saved")

with open(f'{checkpoint_dir}/hd_cell_analysis.pkl', 'wb') as f:
    pickle.dump({'hd_candidates': hd_candidates}, f)
print("✓ HD cell analysis saved")

print(f"\nAll checkpoints saved to: {checkpoint_dir}")
for fn in sorted(os.listdir(checkpoint_dir)):
    size = os.path.getsize(f'{checkpoint_dir}/{fn}') / (1024*1024)
    print(f"  {fn} ({size:.1f} MB)")